# Simple two-output model

In this exercise, you will use the tournament data to build one model that makes two predictions: the scores of both teams in a given game. Your inputs will be the seed difference of the two teams, as well as the predicted score difference from the model you built in chapter 3.

The output from your model will be the predicted score for team 1 as well as team 2. This is called "multiple target regression": one model making more than one prediction.

In [4]:
import pandas as pd
games_season = pd.read_csv("dataset/games_season.csv")
games_season.head()

,season,team_1,team_2,home,score_diff,score_1,score_2,won
0,1985,3745,6664,0,17,81,64,1
1,1985,126,7493,1,7,77,70,1
2,1985,288,3593,1,7,63,56,1
3,1985,1846,9881,1,16,70,54,1
4,1985,2675,10298,1,12,86,74,1


In [5]:
games_tourney = pd.read_csv("dataset/games_tourney.csv")
games_tourney.head()

,season,team_1,team_2,home,seed_diff,score_diff,score_1,score_2,won
0,1985,288,73,0,-3,-9,41,50,0
1,1985,5929,73,0,4,6,61,55,1
2,1985,9884,73,0,5,-4,59,63,0
3,1985,73,288,0,3,9,50,41,1
4,1985,3920,410,0,1,-9,54,63,0


In [3]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
# Define the input
input_tensor = Input(shape=(2,))

# Define the output
output_tensor = Dense(2)(input_tensor)

# Create a model
model = Model(input_tensor, output_tensor)

# Compile the model
model.compile(loss='mean_absolute_error', optimizer='adam')

# Fit a model with two outputs

Now that you've defined your 2-output model, fit it to the tournament data. I've split the data into games_tourney_train and games_tourney_test, so use the training set to fit for now.

This model will use the pre-tournament seeds, as well as your pre-tournament predictions from the regular season model you built previously in this course.

As a reminder, this model will predict the scores of both teams.

In [8]:
# Imports
from tensorflow.keras.layers import Embedding, Flatten, Concatenate
from numpy import unique

# Count the unique number of teams
n_teams = unique(games_season["team_1"]).shape[0]

# Create an embedding layer
team_lookup = Embedding(input_dim=n_teams,
                        output_dim=1,
                        input_length=1,
                        name='Team-Strength')
team_lookup = Embedding(input_dim=n_teams, input_length=1, output_dim=1)

# Create an input layer for the team ID
teamid_in = Input(shape=(1,))

# Lookup the input in the team strength embedding layer
strength_lookup = team_lookup(teamid_in)

# Flatten the output
strength_lookup_flat = Flatten()(strength_lookup)

# Combine the operations into a single, re-usable model
team_strength_model = Model(teamid_in, strength_lookup_flat, name='Team-Strength-Model')

# Create an Input for each team
team_in_1 = Input(shape=(1,), name='Team-1-In')
team_in_2 = Input(shape=(1,), name='Team-2-In')

# Create an input for home vs away
home_in = Input(shape=(1,), name='Home-In')

# Lookup the team inputs in the team strength model
team_1_strength = team_strength_model(team_in_1)
team_2_strength = team_strength_model(team_in_2)

# Combine the team strengths with the home input using a Concatenate layer, then add a Dense layer
out = Concatenate()([team_1_strength, team_2_strength, home_in])
out = Dense(1)(out)

# Make a Model
modelx = Model([team_in_1, team_in_2, home_in], out)

# Compile the model
modelx.compile(optimizer='adam', loss='mean_absolute_error')

# Fit the model to the games_season dataset
modelx.fit([games_season['team_1'], games_season['team_2'], games_season['home']],
          games_season['score_diff'],
          epochs=1,
          verbose=True,
          validation_split=0.1,
          batch_size=2048)

# Predict
games_tourney['pred'] = modelx.predict([games_tourney['team_1'], games_tourney['team_2'], games_tourney['home']])

games_tourney_train = games_tourney

138/138 [==============================] - 0s 2ms/step - loss: 12.0873 - val_loss: 12.1954


In [9]:
# Fit the model
model.fit(games_tourney_train[['seed_diff', 'pred']],
  		  games_tourney_train[['score_1' , 'score_2']],
  		  verbose=True,
  		  epochs=100,
  		  batch_size=16384)

Epoch 1/100
1/1 [==============================] - 0s 1ms/step - loss: 71.1543
Epoch 2/100
1/1 [==============================] - 0s 0s/step - loss: 71.1531
Epoch 3/100
1/1 [==============================] - 0s 997us/step - loss: 71.1520
Epoch 4/100
1/1 [==============================] - 0s 0s/step - loss: 71.1508
Epoch 5/100
1/1 [==============================] - 0s 0s/step - loss: 71.1496
Epoch 6/100
1/1 [==============================] - 0s 992us/step - loss: 71.1485
Epoch 7/100
1/1 [==============================] - 0s 969us/step - loss: 71.1473
Epoch 8/100
1/1 [==============================] - 0s 997us/step - loss: 71.1461
Epoch 9/100
1/1 [==============================] - 0s 855us/step - loss: 71.1450
Epoch 10/100
1/1 [==============================] - 0s 977us/step - loss: 71.1438
Epoch 11/100
1/1 [==============================] - 0s 2ms/step - loss: 71.1426
Epoch 12/100
1/1 [==============================] - 0s 797us/step - loss: 71.1415
Epoch 13/100
1/1 [====================

# Inspect the model (I)

Now that you've fit your model, let's take a look at it. You can use the .get_weights() method to inspect your model's weights.

The input layer will have 4 weights: 2 for each input times 2 for each output.

The output layer will have 2 weights, one for each output.

In [11]:
# Print the model's weights
print(model.get_weights())

# Print the column means of the training data
print(games_tourney_train[['seed_diff', 'pred', 'score_1' , 'score_2']].mean())

[array([[ 0.549266 ,  0.692139 ],
       [-1.1175826,  1.0400342]], dtype=float32), array([0.09999996, 0.09999996], dtype=float32)]
seed_diff     0.000000
pred          0.165515
score_1      71.131318
score_2      71.131318
dtype: float64


# Evaluate the model

Now that you've fit your model and inspected it's weights to make sure it makes sense, evaluate it on the tournament test set to see how well it performs on new data.

In [12]:
games_tourney_test = games_tourney_train

In [13]:
# Evaluate the model on the tournament test data
print(model.evaluate(games_tourney_test[['seed_diff', 'pred']],
  		  games_tourney_test[['score_1' , 'score_2']], verbose=False))

71.03773498535156


# Classification and regression in one model

Now you will create a different kind of 2-output model. This time, you will predict the score difference, instead of both team's scores and then you will predict the probability that team 1 won the game. This is a pretty cool model: it is going to do both classification and regression!

In this model, turn off the bias, or intercept for each layer. Your inputs (seed difference and predicted score difference) have a mean of very close to zero, and your outputs both have means that are close to zero, so your model shouldn't need the bias term to fit the data well.

In [14]:
# Create an input layer with 2 columns
input_tensor = Input(shape=(2,))

# Create the first output
output_tensor_1 = Dense(1, activation='linear', use_bias= False)(input_tensor)

# Create the second output (use the first output as input here)
output_tensor_2 = Dense(1, activation='sigmoid', use_bias=False)(output_tensor_1)

# Create a model with 2 outputs
model = Model(input_tensor, [output_tensor_1, output_tensor_2])

# Compile and fit the model

Now that you have a model with 2 outputs, compile it with 2 loss functions: mean absolute error (MAE) for 'score_diff' and binary cross-entropy (also known as logloss) for 'won'. Then fit the model with 'seed_diff' and 'pred' as inputs. For outputs, predict 'score_diff' and 'won'.

This model can use the scores of the games to make sure that close games (small score diff) have lower win probabilities than blowouts (large score diff).

The regression problem is easier than the classification problem because MAE punishes the model less for a loss due to random chance. For example, if score_diff is -1 and won is 0, that means team_1 had some bad luck and lost by a single free throw. The data for the easy problem helps the model find a solution to the hard problem.

In [16]:
# Import the Adam optimizer
from tensorflow.keras.optimizers import Adam

# Compile the model with 2 losses and the Adam optimzer with a higher learning rate
model.compile(loss=['mean_absolute_error', 'binary_crossentropy'], optimizer=Adam(learning_rate= 0.01))

# Fit the model to the tournament training data, with 2 inputs and 2 outputs
model.fit(games_tourney_train[['seed_diff', 'pred']],
          [games_tourney_train[['score_diff']], games_tourney_train[['won']]],
          epochs=10,
          verbose=True,
          batch_size=16384)

Epoch 1/10
1/1 [==============================] - 0s 2ms/step - loss: 15.8673 - dense_3_loss: 13.1605 - dense_4_loss: 2.7068
Epoch 2/10
1/1 [==============================] - 0s 2ms/step - loss: 15.7409 - dense_3_loss: 13.1173 - dense_4_loss: 2.6236
Epoch 3/10
1/1 [==============================] - 0s 2ms/step - loss: 15.6158 - dense_3_loss: 13.0741 - dense_4_loss: 2.5417
Epoch 4/10
1/1 [==============================] - 0s 1ms/step - loss: 15.4921 - dense_3_loss: 13.0310 - dense_4_loss: 2.4611
Epoch 5/10
1/1 [==============================] - 0s 2ms/step - loss: 15.3698 - dense_3_loss: 12.9880 - dense_4_loss: 2.3818
Epoch 6/10
1/1 [==============================] - 0s 2ms/step - loss: 15.2490 - dense_3_loss: 12.9451 - dense_4_loss: 2.3039
Epoch 7/10
1/1 [==============================] - 0s 2ms/step - loss: 15.1297 - dense_3_loss: 12.9024 - dense_4_loss: 2.2273
Epoch 8/10
1/1 [==============================] - 0s 2ms/step - loss: 15.0120 - dense_3_loss: 12.8598 - dense_4_loss: 2.1523


# Inspect the model (II)

Now you should take a look at the weights for this model. In particular, note the last weight of the model. This weight converts the predicted score difference to a predicted win probability. If you multiply the predicted score difference by the last weight of the model and then apply the sigmoid function, you get the win probability of the game.

In [17]:
# Print the model weights
print(model.get_weights())

# Print the training data means
print(games_tourney_train.mean())

[array([[-0.26558954],
       [-0.26685005]], dtype=float32), array([[1.2994325]], dtype=float32)]
season        2001.193198
team_1        5589.146906
team_2        5589.146906
home             0.000000
seed_diff        0.000000
score_diff       0.000000
score_1         71.131318
score_2         71.131318
won              0.500000
pred             0.165515
dtype: float64


# Inspect the model (II)

Now you should take a look at the weights for this model. In particular, note the last weight of the model. This weight converts the predicted score difference to a predicted win probability. If you multiply the predicted score difference by the last weight of the model and then apply the sigmoid function, you get the win probability of the game.

In [18]:
# Import the sigmoid function from scipy
from scipy.special import expit as sigmoid

# Weight from the model
weight = 0.14

# Print the approximate win probability predicted close game
print(sigmoid(1 * weight))

# Print the approximate win probability predicted blowout game
print(sigmoid(10 * weight))

0.5349429451582145
0.8021838885585818


# Evaluate on new data with two metrics

Now that you've fit your model and inspected its weights to make sure they make sense, evaluate your model on the tournament test set to see how well it does on new data.

Note that in this case, Keras will return 3 numbers: the first number will be the sum of both the loss functions, and then the next 2 numbers will be the loss functions you used when defining the model.

In [19]:
# Evaluate the model on new data
print(model.evaluate(games_tourney_test[['seed_diff', 'pred']],
               [games_tourney_test[['score_diff']], games_tourney_test[['won']]], verbose=False))



[14.6696195602417, 12.733634948730469, 1.9359853267669678]
